In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"  
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
import pennylane as qml
import numpy as np
from numba import njit, vectorize, complex128, int64, void
import timeit
import time
import matplotlib.pyplot as plt
import run

Test with an application of the following set of gates for a system of 5 qubits.

Gate set:
\begin{equation}
H(0) - H(1) - H(2) - H(3) - H(4) - [RX(a, 0) - RX(b, 1) - RX(c, 2) - RX(d, 3) - RX(e, 4) - CZ(0, 1) - CZ(1, 2) - CZ(2, 3) - CZ(3, 4)] * 2
\end{equation}

In [ ]:
def run_circuit_pennylane(angles, num_qubits):
    @qml.qnode(qml.device("lightning.qubit", wires=num_qubits))
    def circuit(angles, num_qubits):
        for i in range(num_qubits):
            qml.Hadamard(wires=i)
        for d in range(len(angles) // num_qubits):
            for i in range(num_qubits):
                qml.RX(angles[num_qubits*d + i], wires=i)
            for i in range(num_qubits - 1):
                qml.CZ(wires=[i, i+1])
        return qml.state()
    return circuit(angles, num_qubits)

In [ ]:
def RX_gate(theta):
    cosine = np.cos(theta / 2)
    sine = np.sin(theta / 2)
    return np.array([[cosine, -1j*sine], [-1j*sine, cosine]], dtype=np.complex128)

def CZ_gate():
    return np.array([[1, 0, 0, 0],
                     [0, 1, 0, 0],
                     [0, 0, 1, 0],
                     [0, 0, 0, -1]], dtype=np.complex128)

def H_gate():
    return (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=np.complex128)

In [ ]:
# @njit
# def update_state(U, state, qubit):
#     new_state = np.zeros_like(state)
#     for g in range(0, len(state), 2**(qubit+1)):
#         for i in range(g, 2**(qubit)+g, 1):
#             new_state[i] = U[0, 0] * state[i] + U[0, 1] * state[i + 2**qubit]
#             new_state[i + 2**qubit] = U[1, 0] * state[i] + U[1, 1] * state[i + 2**qubit]
#     return new_state

# @njit
# def update_state_2q(U, state, qubits):
#     new_state = np.zeros_like(state)
#     q1, q2 = qubits
#     m1 = (1 << q1) - 1
#     m2 = (1 << q2) - 1
#     iters_tot = len(state) >> 2
#     for i in range(iters_tot):
#         i_s1 = (i & m1) | ((i & ~m1) << 1)
#         pos_00 = (i_s1 & m2) | ((i_s1 & ~m2) << 1)
#         pos_01 = pos_00 + (1 << q1)
#         pos_10 = pos_00 + (1 << q2)
#         pos_11 = pos_00 + (1 << q1) + (1 << q2)
#         new_state[pos_00] += U[0,0]*state[pos_00] + U[0,1]*state[pos_01] + U[0,2]*state[pos_10] + U[0,3]*state[pos_11] 
#         new_state[pos_01] += U[1,0]*state[pos_00] + U[1,1]*state[pos_01] + U[1,2]*state[pos_10] + U[1,3]*state[pos_11] 
#         new_state[pos_10] += U[2,0]*state[pos_00] + U[2,1]*state[pos_01] + U[2,2]*state[pos_10] + U[2,3]*state[pos_11] 
#         new_state[pos_11] += U[3,0]*state[pos_00] + U[3,1]*state[pos_01] + U[3,2]*state[pos_10] + U[3,3]*state[pos_11]
#     return new_state

In [ ]:
@njit(void(complex128[:,:], complex128[:], int64, complex128[:]), cache=True, fastmath=True)
def update_state(U, state, qubit, output):
    stride = 1 << qubit
    step = stride << 1
    for g in range(0, len(state), step):
        for i in range(g, g+stride, 1):
            s0 = state[i]
            s1 = state[i + stride]
            output[i] = U[0,0]*s0 + U[0,1]*s1
            output[i + stride] = U[1,0]*s0 + U[1,1]*s1

@njit(void(complex128[:,:], complex128[:], int64, int64, complex128[:]), cache=True, fastmath=True)
def update_state_2q(U, state, q1, q2, output):
    stride1 = 1 << q1
    stride2 = 1 << q2
    step1   = stride1 << 1
    step2   = stride2 << 1
    for g2 in range(0, len(state), step2):
        for g1 in range(g2, g2 + stride2, step1):
            for i in range(g1, g1 + stride1):
                s00 = state[i]
                s01 = state[i + stride1]
                s10 = state[i + stride2]
                s11 = state[i + stride1 + stride2]
                output[i]                    = U[0,0]*s00 + U[0,1]*s01 + U[0,2]*s10 + U[0,3]*s11
                output[i + stride1]          = U[1,0]*s00 + U[1,1]*s01 + U[1,2]*s10 + U[1,3]*s11
                output[i + stride2]          = U[2,0]*s00 + U[2,1]*s01 + U[2,2]*s10 + U[2,3]*s11
                output[i + stride1 + stride2] = U[3,0]*s00 + U[3,1]*s01 + U[3,2]*s10 + U[3,3]*s11

In [ ]:
def run_circuit_bit_manipulation(angles, num_qubits):
    def apply_circuit(angles, num_qubits):
        state = np.zeros(2**num_qubits, dtype=np.complex128)
        state[0] = 1.0 + 0.0j
        buffer = np.empty_like(state)
        for i in range(num_qubits):
            update_state(H_gate(), state, i, buffer)
            state, buffer = buffer, state
        for d in range(len(angles) // num_qubits):
            for i in range(num_qubits):
                update_state(RX_gate(angles[num_qubits*d + i]), state, i, buffer)
                state, buffer = buffer, state
            for i in range(num_qubits - 1):
                update_state_2q(CZ_gate(), state, i, i+1, buffer)
                state, buffer = buffer, state
        return state
    return apply_circuit(angles, num_qubits)

In [ ]:
angles = np.random.normal(0, 2*np.pi, size=(5,))

In [ ]:
x = run_circuit_bit_manipulation(angles, 5)

In [ ]:
xp = run_circuit_pennylane(angles, 5)

In [ ]:
angles_test = [2.65405, -0.814377, -0.2730584, 3.36426]
cpp_out = np.array([
    -0.471954 - 0.205299j,
    0.393229 + 0.283696j,
    -0.377963 - 0.303736j,
    0.323297 + 0.400459j
])
cpp_out = cpp_out.reshape([2]*2).transpose(list(range(2))[::-1]).reshape(-1)
xp = run_circuit_pennylane(angles_test, 2)
print(f"Fidelity: {np.square(np.abs(np.vdot(xp, cpp_out)))}")

In [ ]:
np.linalg.norm(x - xp)

In [ ]:
def test_runtime(max_qubits, title_inclusion, depth=2):
    qubit_list = []
    mean_pennylane = []
    std_pennylane = []
    mean_bitmanip = []
    std_bitmanip = []
    for num_qubits in range(1, max_qubits + 1, 1):
        angles = np.random.uniform(-2*np.pi, 2*np.pi, size=(depth * num_qubits,))
        ns = {**globals(), 'num_qubits': num_qubits, 'angles': angles}
        times_pennylane = timeit.repeat("run_circuit_pennylane(angles, num_qubits)", 
                                        globals=ns,
                                        repeat=100, 
                                        number=100)
        times_bitmanip = timeit.repeat("run_circuit_bit_manipulation(angles, num_qubits)", 
                                       globals=ns,
                                       repeat=100,
                                       number=100)
        mean_pennylane.append(np.mean(times_pennylane) / 100)
        std_pennylane.append(np.std(times_pennylane) / 100)
        mean_bitmanip.append(np.mean(times_bitmanip) / 100)
        std_bitmanip.append(np.std(times_bitmanip) / 100)
        qubit_list.append(num_qubits)
        print(f"Completed {num_qubits} qubits.")
        print(f"Pennylane: {mean_pennylane[-1]} +/- {std_pennylane[-1]} seconds")
        print(f"Bit Manipulation: {mean_bitmanip[-1]} +/- {std_bitmanip[-1]} seconds")
        print("-" * 40)
    fig = plt.figure(figsize=(10, 6))
    for mean, std, label in zip([mean_pennylane, mean_bitmanip], [std_pennylane, std_bitmanip], ["Pennylane", "Bit Manipulation"]):
        plt.plot(qubit_list, mean, label=label)
        plt.fill_between(qubit_list, 
                         np.array(mean) - 2*np.array(std),
                         np.array(mean) + 2*np.array(std),
                         alpha=0.3)
    plt.yscale("log")
    plt.xlabel("Number of Qubits")
    plt.xticks(qubit_list)
    plt.ylabel("Average Runtime (seconds)")
    if title_inclusion:
        plt.title(f"Runtime Comparison - (Depth={depth}) - {title_inclusion}")
    else:
        plt.title(f"Runtime Comparison (Depth={depth})")
    plt.legend()
    plt.grid(True, which="both", ls="--")
    plt.show()

In [ ]:
#TODO: Test the runtime without NJIT and perform the tests by replacing default.qubit with lightning.qubit in pennylane.

In [ ]:
test_runtime(max_qubits=10, depth=20, title_inclusion="Without NJIT")

In [ ]:
d = 10
q = 30
angles = np.random.uniform(-2*np.pi, 2*np.pi, size=(d*q,))
start_time = time.perf_counter()
p_bit = run_circuit_bit_manipulation(angles, q)
end_time = time.perf_counter()
print(f"Bit Manipulation - Execution time: {end_time - start_time} seconds")
start_time = time.perf_counter()
p_pennylane = run_circuit_pennylane(angles, q)
end_time = time.perf_counter()
print(f"Pennylane - Execution time: {end_time - start_time} seconds")
p_bit = p_bit.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
fidelity = np.vdot(p_bit, p_pennylane)
print(f"Fidelity between Bit Manipulation and Pennylane: {fidelity}")

In [ ]:
d = 100
q = 20
angles = np.random.uniform(-2*np.pi, 2*np.pi, size=(d*q,))
p_pennylane = run_circuit_pennylane(angles, q)

In [ ]:
angles = np.loadtxt("angles.txt")

In [ ]:
state_cpp = np.loadtxt("statevector.txt")

In [ ]:
state_test = np.zeros((state_cpp.shape[0]), dtype=np.complex128)
state_test[:] = state_cpp[:, 0] + 1j*state_cpp[:, 1]

In [7]:
import os
os.environ["OMP_NUM_THREADS"] = "8"  
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"

In [1]:
import time
import pennylane as qml
import numpy as np
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

# import run
# import run_gpu
import simulator_gpu

In [ ]:
run_gpu.get_gpu_device_count()

In [2]:
simulator_gpu.get_gpu_count()

1

In [3]:
def run_pennylane(num_qubits, angles):
    @qml.qnode(qml.device("lightning.qubit", wires=num_qubits))
    def circuit(angles, num_qubits):
        for q in range(num_qubits):
            qml.Hadamard(wires=q)
        for d in range(len(angles) // num_qubits):
            for q in range(num_qubits):
                qml.RX(angles[num_qubits*d + q], wires=q)
            for q in range(num_qubits - 1):
                qml.CZ(wires=[q, q+1])
        return qml.state()
    return circuit(angles, num_qubits)

In [6]:
threads = 8
q = 25
depth = 20
print(f"Running with {threads} thread(s) on CPU.")
out_cpu = np.zeros(2**q, dtype=np.complex128)
out_gpu = np.zeros(2**q, dtype=np.complex128)
angles = np.random.uniform(-2*np.pi, 2*np.pi, size=(q*depth,))
t0 = time.perf_counter_ns()
s_pennylane = run_pennylane(q, angles)
t1 = time.perf_counter_ns()
print(f"Pennylane execution time: {(t1 - t0) / 1e9} seconds")
t0 = time.perf_counter_ns()
run.run_circuit_inplace(angles, q, out_cpu, threads)
t1 = time.perf_counter_ns()
print(f"C++ execution time: {(t1 - t0) / 1e9} seconds")
out_cpu = out_cpu.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
fideltiy = np.vdot(out_cpu, s_pennylane)
print(f"Fidelity between C++ implementation and Pennylane: {fideltiy.real**2}")
print(f"Error Norm between C++ implementation and Pennylane: {np.linalg.norm(out_cpu - s_pennylane)}")
# t0 = time.perf_counter_ns()
# run_gpu.run_circuit_gpu(angles, q, out_gpu)
# t1 = time.perf_counter_ns()
# print(f"GPU execution time: {(t1 - t0) / 1e9} seconds")

Running with 8 thread(s) on CPU.
Pennylane execution time: 18.9830167 seconds
C++ execution time: 4.506380452 seconds
Fidelity between C++ implementation and Pennylane: 1.0000000000001745
Error Norm between C++ implementation and Pennylane: 9.111633551232288e-14


In [4]:
def build_instruction(ang, q):
    instruction_list = []
    for i in range(q):
        instruction_list.append(["sx", [i], 0])
        instruction_list.append(["rz", [i], np.pi/2])
        instruction_list.append(["sx", [i], 0])
    depth = len(ang) // q
    for d in range(depth):
        for i in range(q):
            instruction_list.append(["rx", [i], ang[q*d + i]])
        for i in range(q - 1):
            instruction_list.append(["cz", [i, i+1], 0])
    return instruction_list

In [5]:
q = 23
depth = 20
out_gpu = np.zeros(2**q, dtype=np.complex128)
angles = np.random.uniform(-2*np.pi, 2*np.pi, size=(q*depth,))
instruction_list = build_instruction(angles, q)
t0 = time.perf_counter_ns()
s_pennylane = run_pennylane(q, angles)
t1 = time.perf_counter_ns()
print(f"Pennylane execution time: {(t1 - t0) / 1e9} seconds")
t0 = time.perf_counter_ns()
# run_gpu.run_circuit_gpu(angles, q, out_gpu, 8)
simulator_gpu.simulate_circuit(instruction_list, out_gpu, {"num_qubits": q, "noisy": False}, {"num":1}, q, False, 1)
t1 = time.perf_counter_ns()
print(f"GPU execution time: {(t1 - t0) / 1e9} seconds")
out_gpu = out_gpu.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
fideltiy = np.vdot(out_gpu, s_pennylane)
print(f"Fidelity between C++ (GPU) implementation and Pennylane: {fideltiy.real**2}")

Pennylane execution time: 6.575069512 seconds
GPU execution time: 6.890462211 seconds
Fidelity between C++ (GPU) implementation and Pennylane: 2.669132010026169e-08


In [5]:
q = 25
depth = 20
out_gpu = np.zeros(2**q, dtype=np.complex128)
angles = np.random.uniform(-2*np.pi, 2*np.pi, size=(q*depth,))
t0 = time.perf_counter_ns()
run_gpu.run_circuit_gpu(angles, q, out_gpu, 4)
t1 = time.perf_counter_ns()
print(f"GPU execution time: {(t1 - t0) / 1e9} seconds")

GPU execution time: 9.913247061 seconds


In [13]:
out_cpu = out_cpu.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
# out_gpu = out_gpu.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
fidelity = np.vdot(out_cpu, s_pennylane)
# fidelity_gpu = np.vdot(out_gpu, s_pennylane)
print(f"Fidelity computed for C++ implementation with {threads} threads.")
print(f"Fidelity between C++ implementation (CPU) and Pennylane: {fidelity.real**2}")
print(f"Error Norm between C++ implementation (CPU) and Pennylane: {np.linalg.norm(out_cpu - s_pennylane)}")
# print(f"Fidelity between C++ implementation (GPU) and Pennylane: {fidelity_gpu.real**2}")
# print(f"Error Norm between C++ implementation (GPU) and Pennylane: {np.linalg.norm(out_gpu - s_pennylane)}")

Fidelity computed for C++ implementation with 1 threads.
Fidelity between C++ implementation (CPU) and Pennylane: 1.2613047970349966e-09
Error Norm between C++ implementation (CPU) and Pennylane: 1.4141884493566605
